# 02 · Survey, projection and topography

This notebook checks the station locations, chooses the map projection the model is built in,
gets a digital elevation model (DEM) and puts it into that projection. Looking at the stations
and the DEM together is the point: a station on the wrong side of a coastline, an elevation that
doesn't match the DEM, or a DEM that doesn't reach far enough all show up here.

* **Needs:** EDI files; an internet connection for the DEM download (or your own DEM file).
* **Writes** into `work/02_survey/`:
  * `model_frame.json`: the projection; notebooks 03-06 read it
  * the downloaded DEMs, and `topo_big.npz` / `topo_regional.npz`, the detailed and regional DEMs
    clipped and resampled (longitude / latitude)
  * `DEM_projected_gridded.npz`: both DEMs on regular grids in the model frame; notebook 03 reads it
* **Takes:** a minute or two, mostly the download and the regridding.
* **GUI:** setup tabs 1 (*Survey*), 2 (*Topography*) and 3 (*Project DEM*).

## Settings

In [ ]:
EDI_DIR = "sample_data/edis"   # folder of EDIs (or a .lst file)
EXCLUDE = []                   # station names to leave out, e.g. ["CST05", "CST17"]
OUT_DIR = "work/02_survey"

# --- the model frame ------------------------------------------------------------
# "tmerc"  Transverse Mercator centred on the survey (good default)
# "stere"  Oblique Stereographic, "aeqd" Azimuthal Equidistant, "lcc" Lambert Conformal Conic
# "utm"    the UTM zone of the survey centre
# "EPSG:32610" (any EPSG code) a projection of your choice
PROJECTION = "tmerc"
ORIGIN = None                  # (lon, lat) of the model centre, or None for the middle of the stations

# --- the DEM --------------------------------------------------------------------
# Two DEMs: a detailed one around the survey, and a coarse regional one that reaches across the
# whole mesh, padding included. The mesh uses the detailed one wherever it reaches. (One grid
# fine enough for the survey and wide enough for the padding would be far too big, and
# downloads of large areas come at a coarse resolution anyway.)
# Each source is one of:
#   "GMRT": land and sea floor, worldwide, no account needed (good with an ocean)
#   "OpenTopography": land only (SRTM etc.), needs a free API key
#   a path to your own GeoTIFF / NetCDF DEM in longitude / latitude
DEM_SOURCE = "GMRT"            # the detailed DEM
DEM_NEAR_PAD_KM = 50           # how far beyond the stations it reaches (km)
DEM_SAMPLE_ARCSEC = 15         # resampled to this spacing (arc-seconds; 15" is about 460 m)
PROJECTED_SPACING_M = 500      # its grid spacing in the model frame (m)

REGIONAL_DEM_SOURCE = "GMRT"   # the coarse DEM; None for none
# How far beyond the stations the regional DEM reaches (km). It must cover the whole mesh,
# including its padding; notebook 03 checks this. Past the DEM's edge the mesh gets the edge
# values: with an ocean, a short DEM can make the whole padding deep sea.
DEM_PAD_KM = 600
REGIONAL_SAMPLE_ARCSEC = 60    # resampled to this spacing (60" is about 1.8 km)
REGIONAL_SPACING_M = 2000      # its grid spacing in the model frame (m)

OPENTOPO_API_KEY = ""          # only for OpenTopography
OPENTOPO_DEMTYPE = "SRTMGL1"   # only for OpenTopography: SRTMGL1, SRTMGL3, COP30, ...
GRID_METHOD = "linear"         # "linear" or "nearest" interpolation onto that grid

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mtinv_gui.backends import edi_io
from mtinv_gui.backends import projection as P
from mtinv_gui.backends import topography as T

os.makedirs(OUT_DIR, exist_ok=True)

## 1. The stations

A table of every station, then a map. Look for stations far from the rest, two stations in the
same place, or a station at (0, 0): those are header mistakes in the EDI files.

In [ ]:
stations = [st for st in edi_io.read_edi_headers(EDI_DIR) if st.station_id not in set(EXCLUDE)]
table = pd.DataFrame([{"station": st.station_id, "lon": st.lon, "lat": st.lat, "elev_m": st.elev,
                       "periods": st.n_freq, "T_min_s": st.period_min_s, "T_max_s": st.period_max_s}
                      for st in stations])
print(f"{len(stations)} stations (excluded: {EXCLUDE or 'none'})")
spacing = edi_io.station_spacing_geographic(stations)
print(f"nearest-neighbour spacing: min {spacing['min_m'] / 1e3:.1f} km, "
      f"median {spacing['median_m'] / 1e3:.1f} km, max {spacing['max_m'] / 1e3:.1f} km")

same_place = table[table.duplicated(["lon", "lat"], keep=False)]
if len(same_place):
    print("WARNING: stations at the same position:", ", ".join(same_place.station))
if ((table.lon == 0) & (table.lat == 0)).any():
    print("WARNING: stations at (0, 0) -- their EDI headers have no position")
table

In [ ]:
lons, lats = table.lon.to_numpy(), table.lat.to_numpy()
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(lons, lats, "kv", ms=6)
for st in stations:
    ax.annotate(st.station_id, (st.lon, st.lat), fontsize=7, xytext=(3, 3),
                textcoords="offset points")
ax.set_aspect(1 / np.cos(np.radians(lats.mean())))
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("stations")
plt.show()

## 2. The model frame

The model is built in metres, in a map projection centred on the survey. Every projection
stretches distances a little; the table shows by how much over this survey. Under 0.1 % is
negligible. The local projections (`tmerc`, `stere`, `aeqd`, `lcc`) stay small even for large
surveys; UTM grows quickly away from its central meridian.

In [ ]:
bbox = edi_io.bounding_box(stations)         # {"MinLon", "MaxLon", "MinLat", "MaxLat"}
lon0, lat0 = ORIGIN or (0.5 * (bbox["MinLon"] + bbox["MaxLon"]), 0.5 * (bbox["MinLat"] + bbox["MaxLat"]))
lat1, lat2 = P.standard_parallels(bbox["MinLat"], bbox["MaxLat"])   # used by "lcc" only


def make_frame(choice):
    """The model frame for one PROJECTION choice."""
    if choice == "utm":
        return P.make_local_frame("utm", edi_io.suggest_utm(lon0, lat0)[2], lon0, lat0)
    if choice.upper().startswith("EPSG:"):
        return P.make_local_frame("epsg", int(choice.split(":")[1]), lon0, lat0)
    return P.make_local_frame("local", 0, lon0, lat0, choice, lat1, lat2)


rows = []
for choice in ["tmerc", "stere", "aeqd", "lcc", "utm"] + ([PROJECTION] if PROJECTION.upper().startswith("EPSG:") else []):
    d = P.distortion(make_frame(choice), lons, lats)
    rows.append({"choice": choice, "projection": P.frame_label(make_frame(choice)),
                 "worst scale error (%)": round(d["max_err_pct"], 4),
                 "metres per km": round(d["m_per_km"], 2)})
display(pd.DataFrame(rows))

frame = make_frame(PROJECTION)
frame_path = P.save_frame(frame, os.path.join(OUT_DIR, "model_frame.json"))
print("model frame:", P.frame_label(frame))
print("saved", frame_path)

## 3. Get the DEM

The detailed DEM covers the stations plus `DEM_NEAR_PAD_KM`, the regional one the stations plus
`DEM_PAD_KM`. Each is clipped and resampled to its own spacing and saved.

In [ ]:
def km_to_deg(km):
    """Pad in degrees of longitude (the wider one) for a distance in km."""
    return km / (111.2 * np.cos(np.radians(lat0)))


def get_dem(source, pad_km, name):
    """Download (GMRT / OpenTopography) or read (a file path) one DEM."""
    if source in ("GMRT", "OpenTopography"):
        path = T.download(source, bbox, os.path.join(OUT_DIR, f"dem_{source.lower()}_{name}"),
                          pad_deg=km_to_deg(pad_km), api_key=OPENTOPO_API_KEY, demtype=OPENTOPO_DEMTYPE)
        print(f"downloaded {path} ({os.path.getsize(path) / 1e6:.1f} MB)")
    else:
        path = source
    grid = T.read_dem(path)
    res = T.dem_resolution(grid)
    print(f"  {name} DEM: {res['nx']} x {res['ny']} points, {res['dx_m']:.0f} m x {res['dy_m']:.0f} m spacing, "
          f"lon {grid.lon.min():.2f} to {grid.lon.max():.2f}, lat {grid.lat.min():.2f} to {grid.lat.max():.2f}")
    return grid


dem = T.resample_dem(get_dem(DEM_SOURCE, DEM_NEAR_PAD_KM, "near"), bbox,
                     km_to_deg(DEM_NEAR_PAD_KM) * 3600.0, DEM_SAMPLE_ARCSEC)
print(f"detailed: {dem.shape} -> {T.save_dem_npz(dem, os.path.join(OUT_DIR, 'topo_big.npz'))}")
regional = None
if REGIONAL_DEM_SOURCE:
    regional = T.resample_dem(get_dem(REGIONAL_DEM_SOURCE, DEM_PAD_KM, "regional"), bbox,
                              km_to_deg(DEM_PAD_KM) * 3600.0, REGIONAL_SAMPLE_ARCSEC)
    print(f"regional: {regional.shape} -> {T.save_dem_npz(regional, os.path.join(OUT_DIR, 'topo_regional.npz'))}")
print(f"elevations {np.nanmin(dem.z):.0f} to {np.nanmax(dem.z):.0f} m near the survey")

## 4. Stations on the DEM

Both DEMs (the detailed one inside the dashed box), then a close-up of the survey with the 0 m
contour in black. Every station should be where you expect it, on land unless it's a seafloor
station.

In [ ]:
fig, (ax_all, ax_near) = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, pad in ((ax_all, None), (ax_near, 0.3)):
    for grid in ([regional] if regional is not None and pad is None else []) + [dem]:
        im = ax.pcolormesh(grid.lon, grid.lat, grid.z, cmap="terrain", vmin=-4000, vmax=3000, shading="auto")
    ax.contour(dem.lon, dem.lat, dem.z, [0], colors="k", linewidths=0.6)
    if pad is None and regional is not None:
        ax.plot([dem.lon.min(), dem.lon.max(), dem.lon.max(), dem.lon.min(), dem.lon.min()],
                [dem.lat.min(), dem.lat.min(), dem.lat.max(), dem.lat.max(), dem.lat.min()], "k--", lw=0.8)
    ax.plot(lons, lats, "kv", ms=5, mfc="w")
    if pad is not None:
        ax.set_xlim(lons.min() - pad, lons.max() + pad)
        ax.set_ylim(lats.min() - pad, lats.max() + pad)
    ax.set_aspect(1 / np.cos(np.radians(lat0)))
    ax.xaxis.set_major_locator(plt.MaxNLocator(5))
    ax.set_xlabel("longitude")
ax_all.set_ylabel("latitude")
ax_all.set_title("detailed + regional DEM" if regional is not None else "the DEM")
ax_near.set_title("the survey")
fig.colorbar(im, ax=[ax_all, ax_near], label="elevation (m)", shrink=0.8)
plt.show()

Compare each station's elevation in its EDI header with the DEM at that position. Large
differences usually mean a wrong position (or a header with no elevation, which shows as 0 m).

In [ ]:
from scipy.interpolate import RegularGridInterpolator

lat_axis, z_grid = dem.lat, dem.z
if lat_axis[0] > lat_axis[-1]:                     # the interpolator wants ascending axes
    lat_axis, z_grid = lat_axis[::-1], z_grid[::-1]
dem_at = RegularGridInterpolator((lat_axis, dem.lon), z_grid, bounds_error=False)
table["dem_elev_m"] = dem_at(np.c_[lats, lons]).round(1)
table["difference_m"] = (table.elev_m - table.dem_elev_m).round(1)
print(f"EDI minus DEM elevation: median {table.difference_m.median():.0f} m, "
      f"largest {table.difference_m.abs().max():.0f} m")
print("stations below sea level in the DEM:",
      ", ".join(table.station[table.dem_elev_m < 0]) or "none")
table.reindex(table.difference_m.abs().sort_values(ascending=False).index).head(10)[
    ["station", "lon", "lat", "elev_m", "dem_elev_m", "difference_m"]]

## 5. The DEMs in the model frame

Each DEM is regridded onto a regular grid in the model frame (`PROJECTED_SPACING_M` and
`REGIONAL_SPACING_M`), and both go into one file. This is what the mesh is fitted to. In the
plot, (0, 0) is the model centre.

In [ ]:
projected = P.project_dem(dem, frame, PROJECTED_SPACING_M, GRID_METHOD)
if regional is not None:
    projected.regional = P.project_dem(regional, frame, REGIONAL_SPACING_M, GRID_METHOD).topo_cart
pdem_path = P.save_projected_dem(projected, os.path.join(OUT_DIR, "DEM_projected_gridded.npz"))
for name, g in (("detailed", projected.topo_cart), ("regional", projected.regional)):
    if g is not None:    # [..., 0] east (m), [..., 1] north (m), [..., 2] elevation (m)
        print(f"{name}: {g.shape[1]} x {g.shape[0]} grid, east {g[..., 0].min() / 1e3:.0f} to "
              f"{g[..., 0].max() / 1e3:.0f} km, north {g[..., 1].min() / 1e3:.0f} to "
              f"{g[..., 1].max() / 1e3:.0f} km from the model centre")
print("->", pdem_path)

receivers, names = P.project_stations(stations, frame)     # (east, north, 0) in metres
xy = np.asarray(receivers)[:, :2] / 1e3
tc = projected.topo_cart
fig, ax = plt.subplots(figsize=(7, 6))
for g in ([projected.regional] if projected.regional is not None else []) + [tc]:
    im = ax.pcolormesh(g[..., 0] / 1e3, g[..., 1] / 1e3, g[..., 2], cmap="terrain",
                       vmin=-4000, vmax=3000, shading="auto")
ax.contour(tc[..., 0] / 1e3, tc[..., 1] / 1e3, tc[..., 2], [0], colors="k", linewidths=0.6)
ax.plot(xy[:, 0], xy[:, 1], "kv", ms=5, mfc="w")
ax.set_aspect("equal")
ax.set_xlabel("east (km)")
ax.set_ylabel("north (km)")
ax.set_title("projected DEM and stations, model frame")
fig.colorbar(im, ax=ax, label="elevation (m)")
plt.show()

## Next

Notebook `03_mesh_and_receivers` reads `model_frame.json` and `DEM_projected_gridded.npz`.

**For coders:** `projection.make_local_frame` builds the frame (a pyproj CRS plus the origin),
`projection.project_dem` regrids with `scipy.interpolate.griddata`, and the `.npz` holds
`arr_0` (the detailed grid) and `regional`, each of shape (ny, nx, 3): east, north and elevation
in metres. Anything that writes those arrays can replace this notebook's DEMs.